# import preprocessed dataset

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

RND = 42

split raw data into train/val/test

In [2]:
df = pd.read_csv("../01_Preprocessing/out/20newsgroup_preprocessed.csv")
df["text"] = df["text"].fillna("")
# labels_df = df["target"]

df_temp, df_test = train_test_split(
    df,
    test_size=0.2,
    stratify=df["target"],  # maintain class dist.
    random_state=RND,
)

df_train, df_val = train_test_split(
    df_temp, test_size=0.2, stratify=df_temp["target"], random_state=RND
)

X_train = df_train["text"]
y_train = df_train["target"]

X_val = df_val["text"]
y_val = df_val["target"]

X_test = df_test["text"]
y_test = df_test["target"]

# init. vectorizer

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from nltk.tokenize import word_tokenize
from sentence_transformers import SentenceTransformer
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

from vectorization.vectorize import GensimDoc2VecVectorizer, SbertVectorizer

C:\Users\fidel\miniconda3\envs\ML-XAI\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
tf_vec = TfidfVectorizer(lowercase=True, stop_words="english", max_features=10_000)
# d2_vec = Doc2Vec(vector_size=1000, window=5, min_count=2, workers=4)
d2_vec = GensimDoc2VecVectorizer()
# sbert_vec = SentenceTransformer("all-MiniLM-L6-v2")
sbert_vec = SbertVectorizer()  # "all-MiniLM-L6-v2"

vectorizers = {
    #"tfidf": tf_vec,
    #"doc2vec": d2_vec,
    "sbert": sbert_vec,
}

# init. classifiers

In [5]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report
from scipy.stats import randint, uniform
from sklearn.linear_model import LogisticRegression
import numpy as np
import joblib

define search space for hyperparameters

In [6]:
# search space for classifiers in Pipeline
# (param_distributions are given with the 02_Pipeline prefix)
# [parameters are for use of TF-IDF and Doc2Vec]
# search_spaces = {
#     "svm": {
#         "clf": SVC(),
#         "param_distributions": {
#             "clf__C": uniform(0.1, 10),
#             "clf__kernel": ["linear", "rbf"],
#             "clf__gamma": ["scale", "auto"],
#         },
#     },
#     "mlp": {
#         "clf": MLPClassifier(max_iter=400),
#         "param_distributions": {
#             "clf__hidden_layer_sizes": [(50,), (100, 50)],
#             "clf__activation": ["relu", "tanh"],
#             "clf__alpha": uniform(1e-5, 1e-2),
#             "clf__learning_rate": ["constant", "adaptive"],
#         },
#     },
#     "dt": {
#         "clf": DecisionTreeClassifier(),
#         "param_distributions": {
#             "clf__max_depth": randint(3, 20),
#             "clf__min_samples_split": randint(2, 10),
#             "clf__criterion": ["gini", "entropy"],
#         },
#     },
# }


# search space for only classifiers in RandomSearchCV
# (param_distributions are given without the 02_Pipeline prefix)
search_spaces = {
    "svm": {
        "clf": SVC(),
        "param_distributions": {
            "clf__C": uniform(0.1, 10),
            "clf__kernel": ["linear", "rbf"],
            "clf__gamma": ["scale", "auto"],
        },
    },
    "mlp": {
        "clf": MLPClassifier(max_iter=400),
        "param_distributions": {
            "hidden_layer_sizes": [(50,), (100, 50)],
            "activation": ["relu", "tanh"],
            "alpha": uniform(1e-5, 1e-2),
            "learning_rate": ["constant", "adaptive"],
        },
    },
    "dt": {
        "clf": DecisionTreeClassifier(),
        "param_distributions": {
            "max_depth": randint(3, 20),
            "min_samples_split": randint(2, 10),
            "criterion": ["gini", "entropy"],
        },
    },
}

# random search (TF-IDF, Doc2Vec)

In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RND)
best_models = {}

for vec_name, vectorizer in vectorizers.items():
    for clf_name, spec in search_spaces.items():
        name = f"{clf_name}_{vec_name}"  # e.g., svm_tfidf

        pipe = Pipeline([("vectorizer", vectorizer), ("clf", spec["clf"])])

        search = RandomizedSearchCV(
            pipe,
            param_distributions=spec["param_distributions"],
            n_iter=20,
            scoring="f1_weighted",
            cv=cv,
            random_state=RND,
            verbose=1,
            n_jobs=-1,
        )

        print(f"🔍 Running RandomizedSearchCV for {name}...")
        X_train = X_train.reset_index(drop=True)
        y_train = y_train.reset_index(drop=True)
        search.fit(X_train, y_train)
        best_models[name] = search.best_estimator_
        print(f"✅ Best parameters for {name}: {search.best_params_}")

        model_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "models"))
        os.makedirs(model_dir, exist_ok=True)  # create if doesn't exist
        # save model
        joblib.dump(best_models[name], os.path.join(model_dir, f"{name}.pkl"))

🔍 Running RandomizedSearchCV for svm_sbert...
Fitting 3 folds for each of 20 candidates, totalling 60 fits


# random search (SBERT)
[capsulated runs due to non-pipeline functionality of SBERT]

In [11]:
from sklearn.model_selection import RandomizedSearchCV
from vectorization.vectorize import SbertVectorizer

print("Transforming training data with SBERT...")
sbert_vec = SbertVectorizer()  # "all-MiniLM-L6-v2"
vectors = sbert_vec.transform(X_train)
print("Training data transformed with SBERT.")

best_models = {}
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RND)


for name, spec in search_spaces.items():
    print(f"🔍 Running RandomizedSearchCV for {name}...")

    search = RandomizedSearchCV(
        estimator=spec["clf"],
        param_distributions=spec["param_distributions"],
        n_iter=20,
        scoring="f1_weighted",
        cv=cv,
        random_state=RND,
        verbose=1,
        n_jobs=-1,
    )

    search.fit(vectors, y_train)
    best_models[name] = search.best_estimator_
    print(f"✅ Best parameters for {name}: {search.best_params_}")
    print(f"📈 Best score for {name}: {search.best_score_:.4f}\n")

model_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "models"))
os.makedirs(model_dir, exist_ok=True)  # create if doesn't exist
# save model
joblib.dump(best_models[name], os.path.join(model_dir, f"{name}.pkl"), compress=3)

🔍 Running RandomizedSearchCV for svm...
Fitting 3 folds for each of 20 candidates, totalling 60 fits


KeyboardInterrupt: 

# evaluation on validation set

In [ ]:
import os
import joblib
import sys
from sklearn.metrics import classification_report


# Add root of project to import path
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)


# loop models and eval each on val set
model_folder_path = "C://Users//jonas//OneDrive//Desktop//Studium_OvGU//Master//SoSe25//ML//XAI//ML-XAI//models"
print(os.listdir(model_folder_path))
for model in os.listdir(model_folder_path):
    model_path = os.path.join(model_folder_path, model)
    # project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

    # Load model
    loaded_model = joblib.load(model_path)
    print(f"Loaded {model} with params: \n{loaded_model['clf'].get_params()}")
    y_pred = loaded_model.predict(X_val)
    print(f"\n📈 Evaluation report for {model.upper()} on validation set:")
    print(classification_report(y_val, y_pred))
    del loaded_model  # free memory
    # best_models[model[:-4]] = loaded_model  # remove .pkl from name

['dt_doc2vec.pkl', 'dt_tfidf.pkl', 'mlp_doc2vec.pkl', 'mlp_tfidf.pkl', 'svm_doc2vec.pkl', 'svm_tfidf.pkl']
Loaded dt_doc2vec.pkl with params: 
<bound method BaseEstimator.get_params of DecisionTreeClassifier(max_depth=9, min_samples_split=6)>

📈 Evaluation report for DT_DOC2VEC.PKL on validation set:
              precision    recall  f1-score   support

           0       0.06      0.05      0.05       128
           1       0.15      0.20      0.17       156
           2       0.14      0.18      0.16       158
           3       0.19      0.24      0.21       157
           4       0.13      0.15      0.14       154
           5       0.28      0.20      0.24       158
           6       0.34      0.19      0.24       156
           7       0.09      0.23      0.13       158
           8       0.12      0.21      0.16       160
           9       0.44      0.25      0.32       159
          10       0.33      0.30      0.31       160
          11       0.25      0.19      0.22      

# train clfs on train + val

In [ ]:
import os
import joblib
import pandas as pd


X_train_all = pd.concat([X_train, X_val], ignore_index=True)
y_train_all = pd.concat([y_train, y_val], ignore_index=True)


# set paths relative to current notebook (02_Pipeline/pipe.ipynb)
current_dir = os.path.dirname(os.path.abspath("__file__"))  # fallback for notebooks
project_root = os.path.abspath(os.path.join(current_dir, ".."))
models_dir = os.path.join(project_root, "models")
final_models_dir = os.path.join(project_root, "final_models")

# Create output dir if it doesn't exist
os.makedirs(final_models_dir, exist_ok=True)

# 🔁 Load, retrain, and save
for model_file in os.listdir(models_dir):
    if not model_file.endswith(".pkl"):
        continue

    model_path = os.path.join(models_dir, model_file)

    # Load model 02_Pipeline
    model = joblib.load(model_path)

    print(f"🔁 Retraining {model_file} on full training+validation set...")
    model.fit(X_train_all, y_train_all)

    # Save retrained model
    save_path = os.path.join(final_models_dir, model_file)
    joblib.dump(model, save_path)
    print(f"✅ Saved retrained model to: {save_path}\n")

🔁 Retraining dt_doc2vec.pkl on full training+validation set...
✅ Saved retrained model to: c:\Users\jonas\OneDrive\Desktop\Studium_OvGU\Master\SoSe25\ML\XAI\ML-XAI\final_models\dt_doc2vec.pkl

🔁 Retraining dt_tfidf.pkl on full training+validation set...
✅ Saved retrained model to: c:\Users\jonas\OneDrive\Desktop\Studium_OvGU\Master\SoSe25\ML\XAI\ML-XAI\final_models\dt_tfidf.pkl

🔁 Retraining mlp_doc2vec.pkl on full training+validation set...


c:\Users\jonas\OneDrive\Desktop\Studium_OvGU\Master\SoSe25\ML\XAI\ML-XAI\.venv\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (400) reached and the optimization hasn't converged yet.
  warnings.warn(


✅ Saved retrained model to: c:\Users\jonas\OneDrive\Desktop\Studium_OvGU\Master\SoSe25\ML\XAI\ML-XAI\final_models\mlp_doc2vec.pkl

🔁 Retraining mlp_tfidf.pkl on full training+validation set...
✅ Saved retrained model to: c:\Users\jonas\OneDrive\Desktop\Studium_OvGU\Master\SoSe25\ML\XAI\ML-XAI\final_models\mlp_tfidf.pkl

🔁 Retraining svm_doc2vec.pkl on full training+validation set...
✅ Saved retrained model to: c:\Users\jonas\OneDrive\Desktop\Studium_OvGU\Master\SoSe25\ML\XAI\ML-XAI\final_models\svm_doc2vec.pkl

🔁 Retraining svm_tfidf.pkl on full training+validation set...
✅ Saved retrained model to: c:\Users\jonas\OneDrive\Desktop\Studium_OvGU\Master\SoSe25\ML\XAI\ML-XAI\final_models\svm_tfidf.pkl

